# envs

In [1]:
%load_ext autoreload
%autoreload 2

import os
import random
import copy
import torch
from torch import nn
import numpy as np
from torch import optim
from torch.autograd import Variable
from torch.functional import F
from net import *
import argparse
from sklearn.metrics import confusion_matrix
import warnings
from sklearn.model_selection import train_test_split
from collections import Counter
import pickle
import time

from net import *
from utils import *

np.random.seed(0)

seed_torch()
warnings.filterwarnings("ignore")
torch.manual_seed(123)

# functions

In [2]:
def build_model(features):
    model = Net2(features)
    return model

def meta_model_update2(meta_model_1, 
                       vnet_1, 
                       x_1, y_1,
                       epoch, lmbda, lmbda2):

    outputs = meta_model_1(x_1).squeeze(-1)
    outputs = torch.where(torch.isnan(outputs), torch.zeros_like(outputs), outputs)
    cost = nn.BCELoss(reduction='none')
    cost = cost(outputs, y_1)
    cost_v = torch.reshape(cost, (len(cost), 1))
    v_lambda = vnet_1(cost_v.data)
    w1 = abs((v_lambda - v_lambda.max()) / (v_lambda.max() - v_lambda.min()))

    l_f_meta_1 = torch.sum(cost_v * w1) + lmbda * F.l1_loss(meta_model_1.linear.weight,
                                                           target=torch.zeros_like(meta_model_1.linear.weight.detach()),
                                                           size_average=False) + lmbda2 * torch.sum(torch.pow(meta_model_1.linear.weight, 2))

    meta_model_1.zero_grad()
    grads = torch.autograd.grad(l_f_meta_1, (meta_model_1.params()), create_graph=True)
    meta_lr = 1e-3
    meta_model_1.update_params(lr_inner=meta_lr, source_params=grads)
    del grads

def train_auto_meta3(model_1, 
                     vnet_1, 
                     optimizer_model_1, 
                     optimizer_vnet_1, 
                     x_1, y_1, 
                     mx_1, my_1, epoch, lmbda, lmbda2):
    model_1.train()
    my_1 = torch.tensor(my_1)
    y_1 = torch.tensor(y_1)
    
    ###########################################step 1#################################
    featureDim = x_1.shape[1]
    # meta_model_1 = MultiLinearRegression1(featureDim, 1)
    meta_model_1 = build_model(featureDim)
    meta_model_1.load_state_dict(model_1.state_dict())
    meta_model_update2(meta_model_1, vnet_1, x_1, y_1, epoch, lmbda, lmbda2)

    ###########################################step 2#################################
    mx_1 = mx_1.float()
    y_g_hat = meta_model_1(mx_1).squeeze(-1)
    y_g_hat = torch.where(torch.isnan(y_g_hat), torch.zeros_like(y_g_hat), y_g_hat)
    cost = nn.BCELoss(reduction='none')
    l_g_meta1 = cost(y_g_hat, my_1.to(torch.float32))
    
    l_g_meta1 = l_g_meta1 + lmbda* F.l1_loss(meta_model_1.linear.weight,
                                          target=torch.zeros_like(meta_model_1.linear.weight.detach()),
                                          size_average=True)  + lmbda2 * torch.sum(torch.pow(meta_model_1.linear.weight, 2))

    optimizer_vnet_1.zero_grad()
    l_g_meta1.sum().backward()
    optimizer_vnet_1.step()

    ###########################################step 3#################################
    loss_1, w1 = calculate_weight_loss2(model_1, vnet_1, x_1, y_1)

    loss_1 = loss_1 + lmbda * F.l1_loss(model_1.linear.weight,
                                       target=torch.zeros_like(model_1.linear.weight.detach()),
                                       size_average=False) + lmbda2 * torch.sum(torch.pow(model_1.linear.weight, 2))

    optimizer_model_1.zero_grad()
    loss_1.sum().backward()
    optimizer_model_1.step()

def model_train(X_train, y_train, num_epochs, meta_prop, batch_size, batch_size_meta, lmbda, lmbda2):
    sampleNum = X_train.shape[0]
    featureDim = X_train.shape[1]
    meta_data_size = int(sampleNum* meta_prop) + 1 # the metadata size
    
    vnet_1 = VNet(1, 300, 1)
    model_1 = build_model(featureDim)

    criterion_1 = nn.BCELoss(reduction='none')
    optimizer_1 = optim.SGD(model_1.params(), lr=1e-3)

    criterion_vnet_1 = nn.BCELoss(reduction='none')
    optimizer_vnet_1 = torch.optim.Adam(vnet_1.parameters(), 1e-2)

    loss = nn.BCELoss(reduction='none')
    W = torch.normal(0, 0.01, size=(featureDim, 1), requires_grad=True)

    for epoch in range(num_epochs):
        meta_x, meta_y = extract_samples(X_train, y_train, meta_data_size)
        # if epoch == 0:
            # print(Counter(meta_y))
        for batch_indices in data_iter(batch_size, X_train, y_train):
            x_subset = X_train[batch_indices]
            y_subset = y_train[batch_indices]
            y_subset = torch.FloatTensor(y_subset).squeeze(-1) 
            # print("meta_x type:", type(meta_x))
            # print("meta_x shape:", meta_x.shape)
            # print("meta_y type:", type(meta_y))
            # print("meta_y shape:", meta_y.shape)
            # print("meta_y:", Counter(meta_y))
            
            # meta_x_subset, meta_y_subset = next(data_iter_meta(batch_size_meta, meta_x, meta_y)) # the batch_size_meta
            meta_x_subset, meta_y_subset = next(data_iter_meta_class_bal(batch_size_meta, meta_x, meta_y)) # the batch_size_meta

            # print("meta_x_subset type:", type(meta_x_subset))
            # print("meta_x_subset shape:", meta_x_subset.shape)
            # print("meta_y_subset type:", type(meta_y_subset))
            # print("meta_y_subset shape:", meta_y_subset.shape)
            # print("meta_y_subset:", Counter(meta_y_subset))

            train_auto_meta3(model_1, 
                            vnet_1, 
                            optimizer_1, 
                            optimizer_vnet_1, 
                            x_subset, y_subset,
                            meta_x_subset, meta_y_subset, 
                            epoch, lmbda, lmbda2)

            yhat = sigmoid_net(x_subset, W)
            yhat = yhat.squeeze(-1)
            l = loss(yhat, y_subset) + lmbda * F.l1_loss(W, target=torch.zeros_like(W.detach()),
                                                        size_average=False)
            l.sum().backward()
            updater(x_subset.shape[0], W, 1e-3)

    Vnet_test1 = vnet_1(torch.tensor([0.1]).unsqueeze(1).float())
    Vnet_test2 = vnet_1(torch.tensor([100]).unsqueeze(1).float())

    if  Vnet_test1 < Vnet_test2:
        vnet_trend = 'up'
    else:
        vnet_trend = 'down'

    return model_1, optimizer_1, vnet_1, optimizer_vnet_1, vnet_trend


# Train

In [6]:
# Args set
class GetArgs():
    def __init__(self):
        self.DataPath = '../data/'
        self.OutPath = '../result/'
        self.proj = 'exoRBase'
        self.proj1 = 'Benign'
        self.proj2 = 'CRC'
        self.splRat = 0.8
        self.repeat = 1
        self.lmbda = 0.01
        self.lmbda2 = 0.01
        self.num_epochs = 20
        self.meta_prop = 0.15
        self.batch_size = 10 
        self.batch_size_meta = 8
args = GetArgs()

nums = 1
lr = 0.05
lmbda = args.lmbda
lmbda2 = args.lmbda2
num_epochs = args.num_epochs
meta_prop = args.meta_prop 
batch_size = args.batch_size 
batch_size_meta = args.batch_size_meta
OutPath = args.OutPath
proj1 = args.proj1
proj2 = args.proj2

model_path = OutPath + proj1 + '_vs_' + proj2
if not os.path.exists(model_path):
    os.mkdir(model_path)

# cv to choose the best vnet tendency
folds_num = 5
train_folds, X_train, y_train, X_test, y_test = load_data_val_2(args.DataPath, args.proj, args.proj1, args.proj2, args.repeat, 0.8, folds_num)
X_train = torch.tensor(X_train).float()
X_test = torch.tensor(X_test).float()
print(X_train.shape)
print(X_test.shape)
print(Counter(y_train))

start_time = time.time()

better_trend_up = 0
better_trend_dw = 0
for i in range(folds_num):
    # print("Fold")
    # print(i)

    X_train_fold, y_train_fold = train_folds[i]['train']
    X_train_fold = torch.tensor(X_train_fold).float()

    X_val_fold, y_val_fold = train_folds[i]['val']
    X_val_fold = torch.tensor(X_val_fold).float()

    # model training
    # Obtain the data of two different VNET trends
    model_1, optimizer_1, vnet_1, optimizer_vnet_1, vnet_trend_1 = model_train(X_train_fold, y_train_fold, num_epochs, meta_prop, batch_size, batch_size_meta, lmbda, lmbda2)
    vnet_trend_2 = vnet_trend_1
    train_times = 0
    while vnet_trend_1 == vnet_trend_2 and train_times < 50:
        model_2, optimizer_2, vnet_2, optimizer_vnet_2, vnet_trend_2 = model_train(X_train_fold, y_train_fold, num_epochs, meta_prop, batch_size, batch_size_meta, lmbda, lmbda2)
        train_times+=1
        # print(train_times)
    # Compare the two trends
    if train_times < 50:
        best_model = model_select(model_1, model_2, X_val_fold, y_val_fold)
        if best_model == 0:
            vnet_trend = vnet_trend_1
        else:
            vnet_trend = vnet_trend_2
    else:
        vnet_trend = vnet_trend_1
    
    if vnet_trend == 'up':
        better_trend_up += 1
    if vnet_trend == 'down':
        better_trend_dw += 1

# print(better_trend_up)
# print(better_trend_dw)

# validate
if better_trend_up >= better_trend_dw:
    better_trend = 'up'
else:   
    better_trend = 'down'
vnet_trend = 'nan'
while vnet_trend != better_trend:
    model_b, optimizer_b, vnet_b, optimizer_vnet_b, vnet_trend = model_train(X_train_fold, y_train_fold, num_epochs, meta_prop, batch_size, batch_size_meta, lmbda, lmbda2)
end_time = time.time()
training_time = end_time - start_time
print(f"Training time: {training_time:.2f} seconds")

start_time = time.time()
auc, acc, sen, spe,  gmean, f1_score, AUPRC, MCC, balanced_accuracy = model_evaluate(model_b, vnet_b, X_test, y_test, vnet_trend)
end_time = time.time()
Validate_time = end_time - start_time
print(f"Validate time: {Validate_time:.2f} seconds")

# pkl_fils = os.path.join(model_path, f'model.pkl')
# with open(pkl_fils, 'wb') as pickle_file:
#     pickle.dump((model_b, optimizer_b, vnet_b, optimizer_vnet_b, vnet_trend), pickle_file)

torch.Size([132, 3629])
torch.Size([33, 3629])
Counter({0: 104, 1: 28})
Training time: 17.62 seconds
Validate time: 0.01 seconds


In [ ]:
print('{:.3f}\t {:.3f}\t {:.3f}\t {:.3f}\t {:.3f}\t {:.3f}\t {:.3f}\t {:.3f}\t {:.3f}'.
format(auc, acc, sen, spe, gmean, f1_score, AUPRC, MCC, balanced_accuracy))

# effect of meta-data size

In [ ]:
# Args set
meta_prop = 0.15
batch_size_meta_nums = [8, 10, 16]

# fix the meta data set
DataPath = '/disk1/user/liaoshuilin/project/27.Meta_weight/data-20240410-DEG/'
lr = 0.05
lmbda = 0.01
lmbda2 = 0.01
batch_size = 10 
proj = 'PDAC' # PDAC / exoRBase
proj1 = 'I-II' # I-II / Benign
proj2 = 'III-IV' # III-IV / CRC

train_folds, X_train, y_train, X_test, y_test = load_data_val_2(DataPath, proj, proj1, proj2, 1, 0.8, 5)
X_train = torch.tensor(X_train).float()
X_test = torch.tensor(X_test).float()
sampleNum = X_train.shape[0]
featureDim = X_train.shape[1]

meta_data_size = int(sampleNum* meta_prop) + 1 # the metadata size
meta_x, meta_y = extract_samples(X_train, y_train, meta_data_size)

vnet_ori = VNet(1, 300, 1)
model_ori = build_model(featureDim)
# criterion_ori = nn.BCELoss(reduction='none')
optimizer_ori = optim.SGD(model_ori.params(), lr=1e-3)
# criterion_vnet_ori = nn.BCELoss(reduction='none')
optimizer_vnet_ori = torch.optim.Adam(vnet_ori.parameters(), 1e-2)
loss_ori = nn.BCELoss(reduction='none')
W_ori = torch.normal(0, 0.01, size=(featureDim, 1), requires_grad=True)

# evaluate the batch_size_meta
for batch_size_meta in batch_size_meta_nums:
    vnet_1 = vnet_ori
    model_1 = model_ori
    # criterion_1 = criterion_ori
    optimizer_1 = optimizer_ori
    # criterion_vnet_1 = criterion_vnet_ori
    optimizer_vnet_1 = optimizer_vnet_ori
    loss = loss_ori
    W = W_ori
    meta_set = data_iter_meta_class_bal(batch_size_meta, meta_x, meta_y)

    for batch_indices in data_iter(batch_size, X_train, y_train):
        x_subset = X_train[batch_indices]
        y_subset = y_train[batch_indices]
        y_subset = torch.FloatTensor(y_subset).squeeze(-1) 
        meta_x_subset, meta_y_subset = next(meta_set)
        train_auto_meta3(model_1, 
                        vnet_1, 
                        optimizer_1, 
                        optimizer_vnet_1, 
                        x_subset, y_subset,
                        meta_x_subset, meta_y_subset, 
                        1, lmbda, lmbda2)

        yhat = sigmoid_net(x_subset, W)
        yhat = yhat.squeeze(-1)
        l = loss(yhat, y_subset) + lmbda * F.l1_loss(W, target=torch.zeros_like(W.detach()), size_average=False)
        l.sum().backward()
        updater(x_subset.shape[0], W, 1e-3)

    with torch.no_grad():
        y_test_pred = norY(model_1(X_test).squeeze(-1).detach().numpy()) 
        auc, acc, sen, spe,  gmean, f1_score, AUPRC, MCC, balanced_accuracy = print_eva(y_test, y_test_pred, model_1(X_test).squeeze(-1).detach().numpy(), 'test')

    print('{:.3f}\t {:.3f}\t {:.3f}\t {:.3f}\t {:.3f}\t {:.3f}\t {:.3f}\t {:.3f}\t {:.3f}'.
    format(auc, acc, sen, spe, gmean, f1_score, AUPRC, MCC, balanced_accuracy))